In [ ]:
#============================================================
# Experiment 13A: ANN From Scratch
# Synthetic Non-Linear Dataset - Binary Classification
#============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Step 1: Set Random Seed
np.random.seed(42)

# Step 2: Generate Synthetic Dataset
n_samples = 1000

# Generate two input features
X = np.random.randn(n_samples, 2)

# Create non-linear classification problem
# Class 1: x1^2 + x2^2 > 1
# Class 0: x1^2 + x2^2 <= 1
y = (X[:, 0]**2 + X[:, 1]**2 > 1.0).astype(int)

print("Dataset Shape:")
print(X.shape)
print("\nTarget Shape:")
print(y.shape)

# Step 3: Create DataFrame
df = pd.DataFrame({
    "Feature_1": X[:, 0],
    "Feature_2": X[:, 1],
    "Target": y
})
print("\nFirst Five Records:")
print(df.head())

# Step 4: Train-Test Split From Scratch
indices = np.random.permutation(n_samples)
train_size = int(0.80 * n_samples)

train_indices = indices[:train_size]
test_indices = indices[train_size:]

X_train = X[train_indices]
y_train = y[train_indices]
X_test = X[test_indices]
y_test = y[test_indices]

print("\nTraining Samples:", len(X_train))
print("Testing Samples:", len(X_test))

# Step 5: Activation Functions
def sigmoid(x):
    x = np.clip(x, -50, 50)
    return 1 / (1 + np.exp(-x))

def sigmoid_derivative(output):
    return output * (1 - output)

def relu(x):
    return np.maximum(0, x)

def relu_derivative(x):
    return (x > 0).astype(float)

# Step 6: Define ANN Architecture
input_size = 2
hidden_size_1 = 16
hidden_size_2 = 8
output_size = 1

# Step 7: Initialize Weights and Biases
# Input Layer -> Hidden Layer 1
W1 = np.random.randn(input_size, hidden_size_1) * np.sqrt(2 / input_size)
b1 = np.zeros((1, hidden_size_1))

# Hidden Layer 1 -> Hidden Layer 2
W2 = np.random.randn(hidden_size_1, hidden_size_2) * np.sqrt(2 / hidden_size_1)
b2 = np.zeros((1, hidden_size_2))

# Hidden Layer 2 -> Output Layer
W3 = np.random.randn(hidden_size_2, output_size) * np.sqrt(2 / hidden_size_2)
b3 = np.zeros((1, output_size))

# Step 8: Display Network Architecture
print("\n===================================")
print("ANN ARCHITECTURE")
print("===================================")
print("Input Layer:", input_size, "neurons")
print("Hidden Layer 1:", hidden_size_1, "neurons")
print("Hidden Layer 2:", hidden_size_2, "neurons")
print("Output Layer:", output_size, "neuron")

# Step 9: Forward Propagation
def forward_propagation(X):
    # Input -> Hidden Layer 1
    Z1 = np.dot(X, W1) + b1
    A1 = relu(Z1)

    # Hidden Layer 1 -> Hidden Layer 2
    Z2 = np.dot(A1, W2) + b2
    A2 = relu(Z2)

    # Hidden Layer 2 -> Output
    Z3 = np.dot(A2, W3) + b3
    A3 = sigmoid(Z3)

    return Z1, A1, Z2, A2, Z3, A3

# Step 10: Binary Cross-Entropy Loss
def binary_cross_entropy(y_true, y_pred):
    y_pred = np.clip(y_pred, 1e-7, 1 - 1e-7)
    loss = -np.mean(
        y_true.reshape(-1, 1) * np.log(y_pred) +
        (1 - y_true.reshape(-1, 1)) * np.log(1 - y_pred)
    )
    return loss

# Step 11: Backpropagation
def backward_propagation(X, y, Z1, A1, Z2, A2, Z3, A3):
    m = X.shape[0]
    y = y.reshape(-1, 1)

    # Output Layer
    dZ3 = A3 - y
    dW3 = np.dot(A2.T, dZ3) / m
    db3 = np.mean(dZ3, axis=0, keepdims=True)

    # Hidden Layer 2
    dA2 = np.dot(dZ3, W3.T)
    dZ2 = dA2 * relu_derivative(Z2)
    dW2 = np.dot(A1.T, dZ2) / m
    db2 = np.mean(dZ2, axis=0, keepdims=True)

    # Hidden Layer 1
    dA1 = np.dot(dZ2, W2.T)
    dZ1 = dA1 * relu_derivative(Z1)
    dW1 = np.dot(X.T, dZ1) / m
    db1 = np.mean(dZ1, axis=0, keepdims=True)

    return dW1, db1, dW2, db2, dW3, db3

# Step 12: Training Parameters
epochs = 1000
learning_rate = 0.01
loss_history = []
accuracy_history = []

# Step 13: Train ANN
for epoch in range(epochs):

    # Forward Propagation
    Z1, A1, Z2, A2, Z3, A3 = forward_propagation(X_train)

    # Calculate Loss
    loss = binary_cross_entropy(y_train, A3)

    # Calculate Training Accuracy
    predictions = (A3 >= 0.5).astype(int).flatten()
    accuracy = np.mean(predictions == y_train)

    # Backpropagation
    gradients = backward_propagation(X_train, y_train, Z1, A1, Z2, A2, Z3, A3)
    dW1, db1, dW2, db2, dW3, db3 = gradients

    # Update Parameters
    W1 -= learning_rate * dW1
    b1 -= learning_rate * db1
    W2 -= learning_rate * dW2
    b2 -= learning_rate * db2
    W3 -= learning_rate * dW3
    b3 -= learning_rate * db3

    # Store History
    loss_history.append(loss)
    accuracy_history.append(accuracy)

    # Display Progress
    if epoch == 0 or (epoch + 1) % 100 == 0:
        print(f"Epoch {epoch+1:04d} | Loss: {loss:.4f} | Accuracy: {accuracy:.4f}")

# Step 14: Test ANN
Z1_test, A1_test, Z2_test, A2_test, Z3_test, A3_test = forward_propagation(X_test)
test_predictions = (A3_test >= 0.5).astype(int).flatten()
test_accuracy = np.mean(test_predictions == y_test)

# Step 15: Display Test Performance
print("\n===================================")
print("ANN FROM SCRATCH PERFORMANCE")
print("===================================")
print("Test Accuracy:", round(test_accuracy, 4))
print("Test Accuracy (%):", round(test_accuracy * 100, 2))

# Step 16: Actual vs Predicted
test_probabilities = A3_test.flatten()
results = pd.DataFrame({
    "Actual": y_test,
    "Predicted": test_predictions,
    "Probability": test_probabilities
})
print("\nActual vs Predicted:")
print(results.head(20))

# Step 17: Plot Training Loss
plt.figure(figsize=(8, 5))
plt.plot(loss_history)
plt.xlabel("Epoch")
plt.ylabel("Binary Cross-Entropy Loss")
plt.title("ANN From Scratch - Training Loss")
plt.grid(True)
plt.tight_layout()
plt.show()

# Step 18: Plot Training Accuracy
plt.figure(figsize=(8, 5))
plt.plot(accuracy_history)
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("ANN From Scratch - Training Accuracy")
plt.grid(True)
plt.tight_layout()
plt.show()

# Step 19: Actual vs Predicted Plot
number_of_samples = min(50, len(y_test))
sample_number = np.arange(number_of_samples)

plt.figure(figsize=(10, 5))
plt.scatter(sample_number, y_test[:number_of_samples], label="Actual", marker="o")
plt.scatter(sample_number, test_predictions[:number_of_samples], label="Predicted", marker="x")
plt.xlabel("Test Sample Number")
plt.ylabel("Class")
plt.title("ANN From Scratch: Actual vs Predicted")
plt.yticks([0, 1], ["Class 0", "Class 1"])
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# Step 20: Visualize Synthetic Dataset
plt.figure(figsize=(8, 6))
class_0 = (y == 0)
class_1 = (y == 1)
plt.scatter(X[class_0, 0], X[class_0, 1], label="Class 0")
plt.scatter(X[class_1, 0], X[class_1, 1], label="Class 1")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.title("Synthetic Non-Linear Dataset")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# Step 21: Predict a New Sample
new_sample = np.array([[1.2, 0.8]])
_, _, _, _, _, new_probability = forward_propagation(new_sample)
new_prediction = int(new_probability[0, 0] >= 0.5)

# Step 22: Display New Sample Prediction
print("\n===================================")
print("NEW SAMPLE PREDICTION")
print("===================================")
print("Feature 1:", new_sample[0, 0])
print("Feature 2:", new_sample[0, 1])
print("Predicted Probability:", round(new_probability[0, 0], 4))
print("Predicted Class:", new_prediction)

# Step 23: Save Results
results.to_csv("ANN_From_Scratch_Results.csv", index=False)
print("\nResults saved as:")
print("ANN_From_Scratch_Results.csv")

In [ ]:
#============================================================
# Experiment 13B: ANN Classification Using TensorFlow/Keras
# Real-World Dataset: Breast Cancer Wisconsin Dataset
# Dataset available directly from Scikit-learn
#============================================================

# Step 1: Import Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report
)

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

# Step 2: Load Dataset
data = load_breast_cancer()
X = data.data
y = data.target
feature_names = data.feature_names
target_names = data.target_names

print("Dataset Shape:")
print(X.shape)
print("\nTarget Classes:")
print(target_names)

# Step 3: Create DataFrame
df = pd.DataFrame(X, columns=feature_names)
df["Target"] = y
df["Class"] = [target_names[i] for i in y]

print("\nFirst Five Records:")
print(df.head())

# Step 4: Dataset Information
print("\nDataset Information:")
df.info()
print("\nDataset Description:")
print(df.describe())

# Step 5: Class Distribution
print("\n===================================")
print("CLASS DISTRIBUTION")
print("===================================")
print(df["Class"].value_counts())

# Step 6: Separate Features and Target
X = data.data
y = data.target

# Step 7: Train-Test Split (stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print("\nTraining Samples:", X_train.shape[0])
print("Testing Samples:", X_test.shape[0])

# Step 8: Feature Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\nScaled Training Shape:")
print(X_train_scaled.shape)
print("\nScaled Testing Shape:")
print(X_test_scaled.shape)

# Step 9: Create ANN Model
model = Sequential()

# Input + Hidden Layer 1
model.add(Dense(32, activation="relu", input_shape=(X_train_scaled.shape[1],)))

# Hidden Layer 2
model.add(Dense(16, activation="relu"))

# Dropout Layer
model.add(Dropout(0.30))

# Output Layer
model.add(Dense(1, activation="sigmoid"))

# Step 10: Display ANN Architecture
print("\n===================================")
print("ANN MODEL ARCHITECTURE")
print("===================================")
model.summary()

# Step 11: Compile Model
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

# Step 12: Train ANN
history = model.fit(
    X_train_scaled,
    y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.20,
    verbose=1
)

# Step 13: Evaluate Model
test_loss, test_accuracy = model.evaluate(X_test_scaled, y_test, verbose=0)

print("\n===================================")
print("ANN PERFORMANCE")
print("===================================")
print("Test Loss:", round(test_loss, 4))
print("Test Accuracy:", round(test_accuracy, 4))
print("Test Accuracy (%):", round(test_accuracy * 100, 2))

# Step 14: Make Predictions
probabilities = model.predict(X_test_scaled, verbose=0).flatten()
predictions = (probabilities >= 0.5).astype(int)

# Step 15: Actual vs Predicted
results = pd.DataFrame({
    "Actual": y_test,
    "Predicted": predictions,
    "Probability": probabilities
})
print("\n===================================")
print("ACTUAL VS PREDICTED")
print("===================================")
print(results.head(20))

# Step 16: Calculate Accuracy
accuracy = accuracy_score(y_test, predictions)
print("\nAccuracy:", round(accuracy, 4))
print("Accuracy (%):", round(accuracy * 100, 2))

# Step 17: Confusion Matrix
cm = confusion_matrix(y_test, predictions)
print("\n===================================")
print("CONFUSION MATRIX")
print("===================================")
print(cm)

# Step 18: Classification Report
print("\n===================================")
print("CLASSIFICATION REPORT")
print("===================================")
print(classification_report(y_test, predictions, target_names=target_names))

# Step 19: Plot Training Accuracy
plt.figure(figsize=(8, 5))
plt.plot(history.history["accuracy"], marker="o", label="Training Accuracy")
plt.plot(history.history["val_accuracy"], marker="x", label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("ANN Training and Validation Accuracy")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# Step 20: Plot Training Loss
plt.figure(figsize=(8, 5))
plt.plot(history.history["loss"], marker="o", label="Training Loss")
plt.plot(history.history["val_loss"], marker="x", label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("ANN Training and Validation Loss")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# Step 21: Confusion Matrix Visualization
plt.figure(figsize=(7, 5))
plt.imshow(cm, interpolation="nearest")
plt.title("ANN Confusion Matrix")
plt.xlabel("Predicted Class")
plt.ylabel("Actual Class")
plt.xticks([0, 1], target_names)
plt.yticks([0, 1], target_names)
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, cm[i, j], ha="center", va="center")
plt.colorbar()
plt.tight_layout()
plt.show()

# Step 22: Actual vs Predicted Plot
number_of_samples = min(50, len(y_test))
sample_number = np.arange(number_of_samples)

plt.figure(figsize=(10, 5))
plt.scatter(sample_number, y_test[:number_of_samples], label="Actual", marker="o")
plt.scatter(sample_number, predictions[:number_of_samples], label="Predicted", marker="x")
plt.xlabel("Test Sample Number")
plt.ylabel("Class")
plt.title("ANN: Actual vs Predicted")
plt.yticks([0, 1], target_names)
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# Step 23: Predict a New Patient
new_patient = X_test[0].reshape(1, -1)

# Scale New Patient
new_patient_scaled = scaler.transform(new_patient)

# Predict
new_probability = model.predict(new_patient_scaled, verbose=0)[0][0]
new_prediction = int(new_probability >= 0.5)

# Step 24: Display New Patient Prediction
print("\n===================================")
print("NEW PATIENT PREDICTION")
print("===================================")
print("Predicted Probability:", round(new_probability, 4))
print("Predicted Class:", target_names[new_prediction])
print("\nActual Class:", target_names[y_test[0]])

# Step 25: Save Results
results.to_csv("Breast_Cancer_ANN_Results.csv", index=False)
print("\nResults saved as:")
print("Breast_Cancer_ANN_Results.csv")